# Penalty -> Log-Barrier Solver: CPU vs GPU

Compares the numpy/scipy L-BFGS-B (CPU) and torch L-BFGS (GPU)
implementations of the penalty -> barrier solver across 2D and 3D test
cases. Reports neg-Jdet, min Jdet, L2 distortion, and wall time for
each (case, backend) pair.

**Expectation:** GPU has fixed launch overhead (~1-2 s/L-BFGS step), so
small problems (a few thousand voxels) will be slower on GPU; large
problems (~1M+ voxels) win big on GPU. The crossover sits around
10k-100k variables on this hardware (RTX 3050).

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..")))

import os, time
import numpy as np
import matplotlib.pyplot as plt
import torch

from test_cases import make_deformation
from dvfopt import jacobian_det2D, jacobian_det3D, scale_dvf_3d, generate_random_dvf_3d
from dvfopt.core.iterative2d_barrier import iterative_2d_barrier, iterative_2d_barrier_torch
from dvfopt.core.iterative3d_barrier import iterative_3d_barrier
from dvfopt.core.iterative3d_barrier_torch import iterative_3d_barrier_torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
from benchmark_utils import (
    get_output_dir, save_figure, save_results_csv, save_summary_json, log_run_header, log_run_footer, results_to_rows, show_and_save, reset_figure_counter,
)


CUDA available: True
Device: NVIDIA GeForce RTX 3050 OEM


In [2]:
METHOD = "barrier"
NOTEBOOK_NAME = "cpu-vs-gpu"
OUTPUT_DIR = get_output_dir(METHOD, NOTEBOOK_NAME, base="../../output")
reset_figure_counter()
summary = log_run_header(METHOD, NOTEBOOK_NAME, OUTPUT_DIR)


  Benchmark  : cpu-vs-gpu
  Method     : barrier
  Timestamp  : 2026-04-16T01:06:34
  Output dir : ..\..\output\barrier\cpu-vs-gpu


## Helpers

In [3]:
def init_stats_2d(deformation):
    H, W = deformation.shape[-2:]
    phi = np.stack([deformation[1, 0], deformation[2, 0]])  # (2,H,W)
    j = jacobian_det2D(phi)[0]
    return int((j <= 0).sum()), float(j.min()), H * W, (H, W)

def final_stats_2d(phi_out, deformation):
    j = jacobian_det2D(phi_out)[0]
    phi_init = np.stack([deformation[1, 0], deformation[2, 0]])
    return int((j <= 0).sum()), float(j.min()), float(np.linalg.norm(phi_out - phi_init))

def init_stats_3d(dvf):
    j = jacobian_det3D(dvf)
    D, H, W = dvf.shape[1:]
    return int((j <= 0).sum()), float(j.min()), D * H * W, (D, H, W)

def final_stats_3d(phi, dvf):
    j = jacobian_det3D(phi)
    return int((j <= 0).sum()), float(j.min()), float(np.linalg.norm(phi - dvf))

def time_call(fn, *args, **kwargs):
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.time()
    out = fn(*args, **kwargs)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    return out, time.time() - t0

## 2D test cases

Three sizes from `test_cases` covering small (10x10), medium (20x20),
and somewhat larger (20x40) grids.

In [ ]:
CASES_2D = ['01a_10x10_crossing', '03c_20x20_opposite', '01c_20x40_edges']
SEP = '=' * 78
rows_2d = []
for key in CASES_2D:
    print('\n\n' + SEP, flush=True)
    print(f'  2D CASE: {key}', flush=True)
    print(SEP, flush=True)
    deformation, *_ = make_deformation(key)
    init_neg, init_min, n_vox, shape = init_stats_2d(deformation)
    if init_neg == 0:
        print(f'{key}: no negatives, skipping')
        continue

    print('\n-- CPU (windowed) --', flush=True)
    phi_cpu, t_cpu = time_call(iterative_2d_barrier, deformation, verbose=1)
    fn_cpu, fm_cpu, l2_cpu = final_stats_2d(phi_cpu, deformation)

    print('\n-- GPU (windowed) --', flush=True)
    phi_gpu, t_gpu = time_call(iterative_2d_barrier_torch, deformation, verbose=1)
    fn_gpu, fm_gpu, l2_gpu = final_stats_2d(phi_gpu, deformation)

    print('\n-- GPU (full-grid) --', flush=True)
    phi_gpuf, t_gpuf = time_call(iterative_2d_barrier_torch, deformation,
                                 verbose=1, windowed=False)
    fn_gpuf, fm_gpuf, l2_gpuf = final_stats_2d(phi_gpuf, deformation)

    rows_2d.append((key, shape, n_vox, init_neg, init_min,
                    t_cpu, fn_cpu, fm_cpu, l2_cpu,
                    t_gpu, fn_gpu, fm_gpu, l2_gpu,
                    t_gpuf, fn_gpuf, fm_gpuf, l2_gpuf))

print('\n\n' + SEP)
print('  2D SUMMARY')
print(SEP)
print(f'{"case":<25s} {"shape":>10s} {"vox":>6s} {"neg0":>5s} '
      f'{"t_cpu":>8s} {"neg_cpu":>8s} {"L2_cpu":>8s} '
      f'{"t_gpuW":>8s} {"neg_gpuW":>9s} {"L2_gpuW":>8s} '
      f'{"t_gpuF":>8s} {"neg_gpuF":>9s} {"L2_gpuF":>8s} '
      f'{"cpu/gpuW":>9s} {"cpu/gpuF":>9s}')
for r in rows_2d:
    (key, shape, nv, n0, m0, tc, nc, mc, lc,
     tg, ng, mg, lg, tgf, ngf, mgf, lgf) = r
    sh = f'{shape[0]}x{shape[1]}'
    print(f'{key:<25s} {sh:>10s} {nv:>6d} {n0:>5d} '
          f'{tc:>8.2f} {nc:>8d} {lc:>8.4f} '
          f'{tg:>8.2f} {ng:>9d} {lg:>8.4f} '
          f'{tgf:>8.2f} {ngf:>9d} {lgf:>8.4f} '
          f'{tc/tg:>8.2f}x {tc/tgf:>8.2f}x')

## 3D test cases

Synthetic random fields at a few sizes plus the downsampled real volume
(`(132, 80, 114)`, the case that livelocked SLSQP).

In [ ]:
BASE_SHAPE = (3, 3, 3, 3)
MAX_MAGNITUDE = 4.0
SEED = 42
DOWNSAMPLE_FACTORS = [1 / 4, 1 / 3, 1 / 2, 1]
FULL_VOLUME_PATH = '../../../data/corrected_correspondences_count_touching/registered_output/deformation3d.npy'

# GPU full-grid VRAM cap (3 * D * H * W). torch L-BFGS history plus
# cofactor intermediates scale ~30x a single field, so an RTX 3050 (8 GB)
# tops out well before the full 528x320x456 volume.
GPU_FULLGRID_MAX_DOFS = 40_000_000

base_dvf = generate_random_dvf_3d(BASE_SHAPE, MAX_MAGNITUDE, SEED)
synthetic_3d = {}
for size in [(8, 8, 8), (16, 16, 16), (32, 32, 32)]:
    synthetic_3d[f'{size[0]}x{size[1]}x{size[2]}'] = scale_dvf_3d(base_dvf, size)

if os.path.exists(FULL_VOLUME_PATH):
    full = np.load(FULL_VOLUME_PATH)
    _, Df, Hf, Wf = full.shape
    for factor in DOWNSAMPLE_FACTORS:
        label = {1/4: 'real_1_4', 1/3: 'real_1_3', 1/2: 'real_1_2',
                 1: 'real_1_1'}.get(factor, f'real_{factor:.3f}')
        if factor == 1:
            synthetic_3d[label] = full.copy()
        else:
            new_shape = (max(1, int(round(Df * factor))),
                         max(1, int(round(Hf * factor))),
                         max(1, int(round(Wf * factor))))
            synthetic_3d[label] = scale_dvf_3d(full, new_shape)
    del full
    print(f'Loaded real volume -> factors {DOWNSAMPLE_FACTORS}')
else:
    print('Skipping real volume:', FULL_VOLUME_PATH, 'not found')

for k, dvf in synthetic_3d.items():
    n0, m0, nv, _ = init_stats_3d(dvf)
    print(f'  {k:>14s}  vox={nv:>10d}  neg0={n0:>5d}  min0={m0:+.3f}')

In [ ]:
SEP = '=' * 78
rows_3d = []
for key, dvf in synthetic_3d.items():
    init_neg, init_min, n_vox, shape = init_stats_3d(dvf)
    n_dofs = 3 * n_vox
    print('\n\n' + SEP, flush=True)
    print(f'  3D CASE: {key}  shape={shape}  vox={n_vox:,}  '
          f'init_neg={init_neg}  init_min={init_min:+.3f}', flush=True)
    print(SEP, flush=True)

    print('\n-- CPU (windowed) --', flush=True)
    try:
        phi_cpu, t_cpu = time_call(iterative_3d_barrier, dvf, verbose=1)
        fn_cpu, fm_cpu, l2_cpu = final_stats_3d(phi_cpu, dvf)
    except Exception as e:
        print(f'    [ERROR] {type(e).__name__}: {e}', flush=True)
        t_cpu, fn_cpu, fm_cpu, l2_cpu = np.nan, -1, np.nan, np.nan
    else:
        print(f'    CPU:    neg->{fn_cpu}  min->{fm_cpu:+.5f}  '
              f'L2={l2_cpu:.4f}  t={t_cpu:.2f}s', flush=True)

    print('\n-- GPU (windowed) --', flush=True)
    try:
        phi_gpu, t_gpu = time_call(iterative_3d_barrier_torch, dvf, verbose=1)
        fn_gpu, fm_gpu, l2_gpu = final_stats_3d(phi_gpu, dvf)
    except Exception as e:
        print(f'    [ERROR] {type(e).__name__}: {e}', flush=True)
        t_gpu, fn_gpu, fm_gpu, l2_gpu = np.nan, -1, np.nan, np.nan
    else:
        print(f'    GPU-W:  neg->{fn_gpu}  min->{fm_gpu:+.5f}  '
              f'L2={l2_gpu:.4f}  t={t_gpu:.2f}s', flush=True)

    print('\n-- GPU (full-grid) --', flush=True)
    if n_dofs > GPU_FULLGRID_MAX_DOFS:
        print(f'    [SKIPPED] {n_dofs:,} DOFs exceeds cap '
              f'{GPU_FULLGRID_MAX_DOFS:,}', flush=True)
        t_gpuf, fn_gpuf, fm_gpuf, l2_gpuf = np.nan, -1, np.nan, np.nan
    else:
        try:
            phi_gpuf, t_gpuf = time_call(iterative_3d_barrier_torch, dvf,
                                         verbose=1, windowed=False)
            fn_gpuf, fm_gpuf, l2_gpuf = final_stats_3d(phi_gpuf, dvf)
            print(f'    GPU-F:  neg->{fn_gpuf}  min->{fm_gpuf:+.5f}  '
                  f'L2={l2_gpuf:.4f}  t={t_gpuf:.2f}s', flush=True)
        except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
            # torch raises RuntimeError('CUDA out of memory') on some builds
            print(f'    [OOM] {type(e).__name__}: {e}', flush=True)
            t_gpuf, fn_gpuf, fm_gpuf, l2_gpuf = np.nan, -1, np.nan, np.nan
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception as e:
            print(f'    [ERROR] {type(e).__name__}: {e}', flush=True)
            t_gpuf, fn_gpuf, fm_gpuf, l2_gpuf = np.nan, -1, np.nan, np.nan

    rows_3d.append((key, shape, n_vox, init_neg, init_min,
                    t_cpu, fn_cpu, fm_cpu, l2_cpu,
                    t_gpu, fn_gpu, fm_gpu, l2_gpu,
                    t_gpuf, fn_gpuf, fm_gpuf, l2_gpuf))

def _fmt(v, spec):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return f'{"-":>{spec.split(".")[0].lstrip(">").rstrip("df"):s}}'
    return format(v, spec)

print('\n\n' + SEP)
print('  3D SUMMARY')
print(SEP)
print(f'{"case":<14s} {"shape":>14s} {"vox":>10s} {"neg0":>5s} '
      f'{"t_cpu":>9s} {"neg_cpu":>8s} {"L2_cpu":>8s} '
      f'{"t_gpuW":>9s} {"neg_gpuW":>9s} {"L2_gpuW":>8s} '
      f'{"t_gpuF":>9s} {"neg_gpuF":>9s} {"L2_gpuF":>8s}')
for r in rows_3d:
    (key, shape, nv, n0, m0, tc, nc, mc, lc,
     tg, ng, mg, lg, tgf, ngf, mgf, lgf) = r
    sh = f'{shape[0]}x{shape[1]}x{shape[2]}'
    def _f(v, w, dec):
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return f'{"-":>{w}s}'
        return f'{v:>{w}.{dec}f}'
    def _i(v, w):
        if v is None or v == -1:
            return f'{"-":>{w}s}'
        return f'{v:>{w}d}'
    print(f'{key:<14s} {sh:>14s} {nv:>10d} {n0:>5d} '
          f'{_f(tc,9,2)} {_i(nc,8)} {_f(lc,8,4)} '
          f'{_f(tg,9,2)} {_i(ng,9)} {_f(lg,8,4)} '
          f'{_f(tgf,9,2)} {_i(ngf,9)} {_f(lgf,8,4)}')

## Speedup vs problem size

In [ ]:
all_rows = [('2D ' + r[0], r[2], r[5], r[9], r[13]) for r in rows_2d] \
         + [('3D ' + r[0], r[2], r[5], r[9], r[13]) for r in rows_3d]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
labels = [r[0] for r in all_rows]
n_vox = [r[1] for r in all_rows]
t_cpu = [r[2] for r in all_rows]
t_gpu_w = [r[3] for r in all_rows]
t_gpu_f = [r[4] for r in all_rows]
x = np.arange(len(labels)); w = 0.27
ax.bar(x - w, t_cpu, w, label='CPU windowed', color='tab:blue')
ax.bar(x, t_gpu_w, w, label='GPU windowed', color='tab:orange')
ax.bar(x + w, t_gpu_f, w, label='GPU full-grid', color='tab:green')
ax.set_yscale('log')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Time (s)'); ax.set_title('Wall time (log scale)')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')

ax = axes[1]
sp_gpu_w = [c / g for c, g in zip(t_cpu, t_gpu_w)]
sp_gpu_f = [c / g for c, g in zip(t_cpu, t_gpu_f)]
width = 0.4
ax.bar(x - width/2, sp_gpu_w, width, label='CPU-win / GPU-win', color='tab:orange')
ax.bar(x + width/2, sp_gpu_f, width, label='CPU-win / GPU-full', color='tab:green')
ax.axhline(1.0, color='black', linestyle='--', linewidth=0.7)
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Speedup vs CPU')
ax.set_title('Speedup factor (>1 = GPU faster)')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout(); show_and_save(OUTPUT_DIR)

# Scatter: label each case once at the midpoint of its two points and
# offset odd/even cases above/below to avoid overlap.
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.scatter(n_vox, sp_gpu_w, s=80, edgecolor='black',
           color='tab:orange', label='GPU windowed', zorder=3)
ax.scatter(n_vox, sp_gpu_f, s=80, edgecolor='black',
           color='tab:green', label='GPU full-grid', zorder=3)

for i, (x_, lab) in enumerate(zip(n_vox, labels)):
    y_w = sp_gpu_w[i]
    y_f = sp_gpu_f[i]
    ys = [y for y in (y_w, y_f) if y is not None and np.isfinite(y)]
    if not ys:
        continue
    y_anchor = max(ys)
    dy = 12 if i % 2 == 0 else -18
    ax.annotate(lab, (x_, y_anchor), fontsize=8,
                textcoords='offset points', xytext=(6, dy),
                arrowprops=dict(arrowstyle='-', color='gray',
                                lw=0.5, alpha=0.6))

ax.set_xscale('log')
ax.axhline(1.0, color='black', linestyle='--', linewidth=0.7)
ax.set_xlabel('Total voxels'); ax.set_ylabel('Speedup (CPU / GPU)')
ax.set_title('Speedup vs problem size')
ax.margins(y=0.25)
ax.legend(loc='upper left'); ax.grid(True, alpha=0.3)
plt.tight_layout(); show_and_save(OUTPUT_DIR)

In [8]:
# --- Save results ---
COLUMNS = ["case", "shape", "n_vox", "n_neg_init", "min_jdet_init",
           "t_cpu", "n_neg_cpu", "min_jdet_cpu", "l2_cpu",
           "t_gpu_win", "n_neg_gpu_win", "min_jdet_gpu_win", "l2_gpu_win",
           "t_gpu_full", "n_neg_gpu_full", "min_jdet_gpu_full", "l2_gpu_full"]

def _tuple_to_row(r):
    (key, shape, nv, n0, m0,
     tc, nc, mc, lc,
     tg, ng, mg, lg,
     tgf, ngf, mgf, lgf) = r
    sh = "x".join(str(s) for s in shape)
    return [key, sh, nv, n0, round(m0, 6),
            round(tc, 4), nc, round(mc, 6), round(lc, 6),
            round(tg, 4), ng, round(mg, 6), round(lg, 6),
            round(tgf, 4), ngf, round(mgf, 6), round(lgf, 6)]

if rows_2d:
    save_results_csv([_tuple_to_row(r) for r in rows_2d],
                     COLUMNS, OUTPUT_DIR, name="results_2d")
if rows_3d:
    save_results_csv([_tuple_to_row(r) for r in rows_3d],
                     COLUMNS, OUTPUT_DIR, name="results_3d")

# Build a combined dict keyed by case label for log_run_footer + summary.
combined = {}
def _add(tag, key, n0, m0, t, nf, mf, lf):
    combined[f"{tag} {key}"] = {
        "n_neg_init": n0, "n_neg_final": nf,
        "min_jdet_init": m0, "min_jdet": mf,
        "l2_err": lf, "time": t,
    }
for r in rows_2d:
    (key, shape, nv, n0, m0, tc, nc, mc, lc,
     tg, ng, mg, lg, tgf, ngf, mgf, lgf) = r
    _add("2D cpu-win",  key, n0, m0, tc, nc, mc, lc)
    _add("2D gpu-win",  key, n0, m0, tg, ng, mg, lg)
    _add("2D gpu-full", key, n0, m0, tgf, ngf, mgf, lgf)
for r in rows_3d:
    (key, shape, nv, n0, m0, tc, nc, mc, lc,
     tg, ng, mg, lg, tgf, ngf, mgf, lgf) = r
    _add("3D cpu-win",  key, n0, m0, tc, nc, mc, lc)
    _add("3D gpu-win",  key, n0, m0, tg, ng, mg, lg)
    _add("3D gpu-full", key, n0, m0, tgf, ngf, mgf, lgf)

if combined:
    summary = log_run_footer(summary, combined)
save_summary_json(summary, OUTPUT_DIR)


  [saved] ..\..\output\barrier\cpu-vs-gpu\results_2d.csv
  [saved] ..\..\output\barrier\cpu-vs-gpu\results_3d.csv

------------------------------------------------------------------------
  Cases: 21  |  Converged: 21/21  |  Total time: 468.57s
------------------------------------------------------------------------
  [saved] ..\..\output\barrier\cpu-vs-gpu\summary.json
